In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [2]:
df = pd.read_parquet('dinamika-tsen-obyavlenii_ru_8dolfy.parquet')

df.head()

,period,value,price_type,source,freq,ref_area,decimals,unit_measure,unit_mult
0,2022-01-01,185198.830938,"Первичный рынок, бизнес-класс",Данные Домклик,Месяц,Республика Татарстан,2,руб. за квадратный метр,0
1,2022-02-01,188588.001063,"Первичный рынок, бизнес-класс",Данные Домклик,Месяц,Республика Татарстан,2,руб. за квадратный метр,0
2,2022-03-01,199350.688425,"Первичный рынок, бизнес-класс",Данные Домклик,Месяц,Республика Татарстан,2,руб. за квадратный метр,0
3,2022-04-01,207176.597970,"Первичный рынок, бизнес-класс",Данные Домклик,Месяц,Республика Татарстан,2,руб. за квадратный метр,0
4,2022-05-01,208417.034438,"Первичный рынок, бизнес-класс",Данные Домклик,Месяц,Республика Татарстан,2,руб. за квадратный метр,0


In [3]:
# Change column names as desired
df = df.rename(columns={'period': 'date', 'price_type': 'category'})

In [4]:
# Explore the main characteristics of the data
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10531 entries, 0 to 10530
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          10531 non-null  object 
 1   value         10531 non-null  float64
 2   category      10531 non-null  object 
 3   source        10531 non-null  object 
 4   freq          10531 non-null  object 
 5   ref_area      10531 non-null  object 
 6   decimals      10531 non-null  object 
 7   unit_measure  10531 non-null  object 
 8   unit_mult     10531 non-null  object 
dtypes: float64(1), object(8)
memory usage: 740.6+ KB


date            0
value           0
category        0
source          0
freq            0
ref_area        0
decimals        0
unit_measure    0
unit_mult       0
dtype: int64

In [5]:
# None objects are missing
# Change 'date' column type from object to datetime
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# Add columns with separate year and month information for further analysis
df['month'] = df['date'].dt.month

In [6]:
df.describe()

,value,year,month
count,10531.000000,10531.000000,10531.000000
mean,121582.607351,2023.394075,6.200836
std,57845.389812,1.081336,3.351407
min,49571.277121,2022.000000,1.000000
25%,84833.855668,2022.000000,3.000000
50%,106142.551294,2023.000000,6.000000
75%,138672.266210,2024.000000,9.000000
max,564013.076609,2025.000000,12.000000


In [7]:
# Consider unique elements
pd.DataFrame(data=[(column, df[column].nunique(), df[column].unique()) for column in df.columns],
             columns=['Столбец', 'Количество уникальных элементов', 'Уникальные элементы'])

,Столбец,Количество уникальных элементов,Уникальные элементы
0,date,45,"[2022-01-01T00:00:00.000000000, 2022-02-01T00:00:00.000000000, 2022-03-01T00:00:00.000000000, 20..."
1,value,8408,"[185198.8309380862, 188588.0010634082, 199350.6884251968, 207176.5979699248, 208417.0344382022, ..."
2,category,4,"[Первичный рынок, бизнес-класс, Вторичный рынок, Первичный рынок, Первичный рынок, эконом/комфорт]"
3,source,1,[Данные Домклик]
4,freq,1,[Месяц]
5,ref_area,84,"[Республика Татарстан, Псковская область, Московская область, Ростовская область, Республика Сах..."
6,decimals,1,[2]
7,unit_measure,1,[руб. за квадратный метр]
8,unit_mult,1,[0]
9,year,4,"[2022, 2023, 2024, 2025]"


In [8]:
# Get rid of the generalized housing category 'Первичный рынок'
df = df[df['category'] != 'Первичный рынок']

# Remove columns that contain only one value
df = df.drop(columns=[column for column in df.columns if df[column].nunique() == 1])

# Check the presence of the 'Россия' region in the 'ref_area' field
'Россия' in df['ref_area'].unique()

True

In [9]:
# Deliberately delete ready-made data on Russia in order to practice in case there is no such data one day
df.drop(df[df['ref_area'] == 'Россия'].index, inplace=True)

In [10]:
# Check negative prices (for training)
assert (df['value'] > 0).all(), "Обнаружены отрицательные значения цен"

In [11]:
# Get rid of duplicates just in case
df = df.drop_duplicates()

# Update indexing
df.reset_index(drop=True, inplace=True)

df.head()

,date,value,category,ref_area,year,month
0,2022-01-01,185198.830938,"Первичный рынок, бизнес-класс",Республика Татарстан,2022,1
1,2022-02-01,188588.001063,"Первичный рынок, бизнес-класс",Республика Татарстан,2022,2
2,2022-03-01,199350.688425,"Первичный рынок, бизнес-класс",Республика Татарстан,2022,3
3,2022-04-01,207176.597970,"Первичный рынок, бизнес-класс",Республика Татарстан,2022,4
4,2022-05-01,208417.034438,"Первичный рынок, бизнес-класс",Республика Татарстан,2022,5


In [12]:
df.to_csv('dynamics of prices per square meter.csv', index=False)

In [13]:
df.to_parquet('dynamics of prices per square meter.parquet', index=False)